In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
import joblib


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Sto addestrando su: {device}")

Sto addestrando su: mps


In [3]:
# ==========================================
# 1. CARICAMENTO E PREPARAZIONE DATI
# ==========================================
df = pd.read_csv('AI_machine_learning_data_set.csv')
    
# Rimuoviamo le colonne non necessarie per il modello
df_model = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud', 'step'])

# Separazione feature (X) e target (y)
X = df_model.drop('isFraud', axis=1)

y = df_model['isFraud'] 

# Train-Test Split (30% test, stratificato per mantenere la percentuale di frodi)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


# UNDER-SAMPLING (BILANCIAMENTO TRAINING SET)

# Uniamo momentaneamente X_train e y_train
df_train_completo = pd.concat([X_train, y_train], axis=1)

# Separiamo le frodi dalle transazioni lecite
frodi_train = df_train_completo[df_train_completo['isFraud'] == 1]
non_frodi_train = df_train_completo[df_train_completo['isFraud'] == 0]

# Under-sampling: prendiamo casualmente tante lecite quante sono le frodi
non_frodi_bilanciate = non_frodi_train.sample(n=len(frodi_train), random_state=42)

# Creiamo il nuovo set bilanciato e lo mescoliamo (sample frac=1)
df_train_bilanciato = pd.concat([frodi_train, non_frodi_bilanciate]).sample(frac=1, random_state=42)

# Riassegniamo X e y per l'addestramento della Rete Neurale
x_train_bilanciato = df_train_bilanciato.drop('isFraud', axis=1)
y_train_bilanciato = df_train_bilanciato['isFraud']

print("--- CONTEGGIO BILANCIATO TRAINING SET (DEEP LEARNING) ---")
print(y_train_bilanciato.value_counts())


# PREPROCESSING

categorical_cols = ['type']
numeric_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)


X_train_processed = preprocessor.fit_transform(x_train_bilanciato)

# Il test set rimane intatto (come è giusto che sia nel mondo reale)
X_test_processed = preprocessor.transform(X_test)

--- CONTEGGIO BILANCIATO TRAINING SET (DEEP LEARNING) ---
isFraud
1    5749
0    5749
Name: count, dtype: int64


In [4]:
# ==========================================
# 2. CONVERSIONE IN TENSORI PYTORCH
# ==========================================
import torch
from torch.utils.data import TensorDataset, DataLoader

# CONVERSIONE IN MATRICI DENSE

# Se X_train_processed è una matrice sparsa, la "scompattiamo" con .toarray()
if hasattr(X_train_processed, "toarray"):
    X_train_dense = X_train_processed.toarray()
else:
    X_train_dense = X_train_processed

if hasattr(X_test_processed, "toarray"):
    X_test_dense = X_test_processed.toarray()
else:
    X_test_dense = X_test_processed


# CREAZIONE DEI TENSORI PYTORCH

# X e Y del TRAIN (Usiamo i dati bilanciati!)
X_train_tensor = torch.tensor(X_train_dense, dtype=torch.float32)
# Aggiungiamo .values a y_train_bilanciato per estrarre l'array puro
y_train_tensor = torch.tensor(y_train_bilanciato.values, dtype=torch.float32).unsqueeze(1)

# X e Y del TEST (Usiamo i dati originali del test)
X_test_tensor = torch.tensor(X_test_dense, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)


# CREAZIONE DEL DATALOADER

batch_size = 2048
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print("Tensori e DataLoader creati con successo!")
print(f"Dimensioni X_train_tensor: {X_train_tensor.shape}")
print(f"Dimensioni y_train_tensor: {y_train_tensor.shape}")

Tensori e DataLoader creati con successo!
Dimensioni X_train_tensor: torch.Size([11498, 9])
Dimensioni y_train_tensor: torch.Size([11498, 1])


In [5]:
# ==========================================
# 3. DEFINIZIONE DELLA RETE NEURALE
# ==========================================
class FraudDetectionNN(nn.Module):
    def __init__(self, input_dim):
        super(FraudDetectionNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2), # Dropout moderato
            
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.1), # Dropout più leggero verso la fine
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.network(x)

input_dim = X_train_tensor.shape[1]
model = FraudDetectionNN(input_dim)
model.to(device)

FraudDetectionNN(
  (network): Sequential(
    (0): Linear(in_features=9, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=16, out_features=1, bias=True)
  )
)

In [6]:
# ==========================================
# 4. CONFIGURAZIONE ADDESTRAMENTO
# ==========================================
# GESTIONE SBILANCIAMENTO: Equivalente a class_weight="balanced"
# Loss Function: Manteniamo BCEWithLogitsLoss (senza pesi aggiuntivi)
criterion = nn.BCEWithLogitsLoss()

# Optimizer: Manteniamo Adam con il learning rate che avevi scelto
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Configurazione completata: Loss Function senza pesi artificiali.")

Configurazione completata: Loss Function senza pesi artificiali.


In [7]:
# ==========================================
# 5. TRAINING LOOP
# ==========================================
epochs = 100

print("Inizio Addestramento...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoca [{epoch+1}/{epochs}], Loss Media: {avg_loss:.4f}")

Inizio Addestramento...
Epoca [1/100], Loss Media: 0.7015
Epoca [2/100], Loss Media: 0.6953
Epoca [3/100], Loss Media: 0.6894
Epoca [4/100], Loss Media: 0.6855
Epoca [5/100], Loss Media: 0.6806
Epoca [6/100], Loss Media: 0.6730
Epoca [7/100], Loss Media: 0.6687
Epoca [8/100], Loss Media: 0.6630
Epoca [9/100], Loss Media: 0.6570
Epoca [10/100], Loss Media: 0.6524
Epoca [11/100], Loss Media: 0.6482
Epoca [12/100], Loss Media: 0.6430
Epoca [13/100], Loss Media: 0.6370
Epoca [14/100], Loss Media: 0.6342
Epoca [15/100], Loss Media: 0.6302
Epoca [16/100], Loss Media: 0.6258
Epoca [17/100], Loss Media: 0.6184
Epoca [18/100], Loss Media: 0.6174
Epoca [19/100], Loss Media: 0.6105
Epoca [20/100], Loss Media: 0.6093
Epoca [21/100], Loss Media: 0.6054
Epoca [22/100], Loss Media: 0.6010
Epoca [23/100], Loss Media: 0.5976
Epoca [24/100], Loss Media: 0.5902
Epoca [25/100], Loss Media: 0.5885
Epoca [26/100], Loss Media: 0.5838
Epoca [27/100], Loss Media: 0.5789
Epoca [28/100], Loss Media: 0.5778
Epoca

In [8]:
# ==========================================
# 6. VALUTAZIONE DEL MODELLO
# ==========================================
model.eval()  # Mettiamo il modello in modalità valutazione (disabilita Dropout/BatchNorm se presenti)

X_test_device = X_test_tensor.to(device)
y_test_device = y_test_tensor.to(device)

with torch.no_grad():
    # 1. Otteniamo le predizioni sui dati di test
    test_outputs = model(X_test_device)
    
    # 2. Applichiamo la Sigmoide per avere probabilità tra 0 e 1
    probabilities = torch.sigmoid(test_outputs)
    
    # 3. Applichiamo la soglia di 0.5 per ottenere le classi predette (0 o 1)
    y_pred_tensor = (probabilities >= 0.8).float()
    
    # 4. CALCOLO ACCURATEZZA FINALE
    corrette = (y_pred_tensor == y_test_device).sum().item()
    totali = y_test_device.size(0)
    accuratezza_finale = corrette / totali
    print(f"\nACCURATEZZA GLOBALE SUL TEST SET: {accuratezza_finale:.2%}")
    
    # Per scikit-learn (matrice confusione/report) servono array numpy su CPU e 1D
    y_pred = y_pred_tensor.cpu().numpy().astype(int).ravel()

print(f"\nAccuratezza Finale sul Test Set: {accuratezza_finale * 100:.2f}%")

#matrice di confusione
cm = confusion_matrix(y_test, y_pred)

#DataFrame con etichette per righe e colonne
cm_con_titoli = pd.DataFrame(cm, 
                             columns=['PREDETTO LECITO', 'PREDETTO FRODE'], 
                             index=['REALE LECITO', 'REALE FRODE'])

#Stampa a schermo della matrice formattata
print("\nMATRICE DI CONFUSIONE:")
print(cm_con_titoli)

print("\nReport di Classificazione:")
print(classification_report(y_test, y_pred))




ACCURATEZZA GLOBALE SUL TEST SET: 99.88%

Accuratezza Finale sul Test Set: 99.88%

MATRICE DI CONFUSIONE:
              PREDETTO LECITO  PREDETTO FRODE
REALE LECITO          1905586             736
REALE FRODE              1476             988

Report di Classificazione:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906322
           1       0.57      0.40      0.47      2464

    accuracy                           1.00   1908786
   macro avg       0.79      0.70      0.74   1908786
weighted avg       1.00      1.00      1.00   1908786



In [9]:
# ==========================================
# 7. SALVATAGGIO MODELLO E PREPROCESSOR
# ==========================================
import joblib
import torch

# 1. Salva il preprocessore (che contiene lo StandardScaler e il OneHotEncoder bilanciati)
joblib.dump(preprocessor, 'preprocessor.pkl')

# 2. Salva i pesi della Rete Neurale addestrata
torch.save(model.state_dict(), 'fraud_detection_pytorch.pth')

print("✅ File 'preprocessor.pkl' e 'fraud_detection_pytorch.pth' creati e salvati con successo!")

✅ File 'preprocessor.pkl' e 'fraud_detection_pytorch.pth' creati e salvati con successo!
